# Final longitudinal OTP-versus-counterpick datasets

Execute sequentially. Inputs are pinned to longitudinal snapshot `20261018T195205790466Z` and LoLalytics run `20261020T164500_repaired`. This notebook prepares and validates versioned datasets; it performs no EDA, model fitting, threshold selection or graphics.

The only optional network work is one bounded retry of up to 12 unresolved NA pages through the existing helper. A recorded prior retry prevents repetition. Successful pages are reused; failures remain explicit and do not block offline preparation. No Riot API calls or full collection are started.

The master retains every target. Candidate rows are comparisons, not extra observed outcomes. Only actual picks have an outcome label. Retrospective external rates and completed-match role inference remain limitations even with time-safe player histories.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
ROOT = next(p for p in [start, *start.parents] if (p/'hlr_model/longitudinal_pipeline.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import importlib
import hlr_model.final_dataset_helpers as prep
importlib.reload(prep)
print('Pinned inputs:', prep.SNAPSHOT, prep.COLLECTION)


Pinned inputs: 20261018T195205790466Z 20261020T164500_repaired


In [2]:
cleanup = prep.optional_collection_cleanup(ROOT)
print('Limited cleanup:', cleanup['action'])
if cleanup.get('last_retry_session'):
    print('Prior bounded retry:', cleanup['last_retry_session']['mode'],
          '| pause:', cleanup['last_retry_session'].get('pause_reason'))


Limited cleanup: skipped_previously_retried
Prior bounded retry: retry_unresolved_only | pause: Repeated/systemic page failures: 5 consecutive; 5/5 in recent regional pages. Inspect cached evidence.


In [3]:
data = prep.load_inputs(ROOT)
validation = prep.validate_sources(data)
print('Input rows:', {name: len(data[name]) for name in
      ['longitudinal_modeling', 'historical_matches', 'target_matches', 'matchup_lookup']})
print('Source validation: PASS')


Input rows: {'longitudinal_modeling': 7238, 'historical_matches': 29889, 'target_matches': 7238, 'matchup_lookup': 35994}
Source validation: PASS


Candidate eligibility comes from the saved regional decision mapping, deduplicated across regions and checked against strictly pregame histories. The selected champion is included once even with zero experience or unidentified opponent. Alternative eligibility requires prior play in the target role and excludes the known opponent's champion. Draft availability is unverified.

Both all-role and same-role experience use initial preserved history plus earlier target games whose end time is strictly before kickoff. Both main definitions use the highest prior count with smallest champion ID as the tie rule. The original all-role main is preserved. Raw historical WR is wins/games, missing at zero games; the separately named Beta(1,1) mean is `(wins+1)/(games+2)`.

Regional raw population rates are percentages. Explicit valid-raw probability columns divide by 100. Website Delta 2 and derived tier-centered values remain separately named; the latter are not clipped or treated as probabilities. Main and observed-pick comparisons use the same role, opponent and region.

In [4]:
candidates, decision_features = prep.build_candidates(data)
tables = prep.assemble(data, candidates, decision_features)
validation.update(prep.validate_outputs(data, tables))
counts = prep.summary_counts(data, tables)
display(pd.DataFrame({name: counts[name] for name in tables}).T)
print('Observed coverage — Global:', counts['observed_global_valid_rows'],
      '| NA:', counts['observed_na_valid_rows'], '| both:', counts['observed_both_regions_valid_rows'])
print('Exclusions from primary:', counts['global_exclusion_combinations'])
print('Exclusions from paired:', counts['paired_exclusion_combinations'])
print('Validation:', {key: value for key, value in validation.items() if value == 'PASS'})


,rows,columns,players,unique_matches
observed_player_games,7238,269,185,6806
candidate_champion_comparisons,83523,285,185,6806
observed_global_analysis,7227,269,184,6797
observed_paired_region_analysis,7214,269,184,6784
observed_exclusions,24,269,10,24


Observed coverage — Global: 7232 | NA: 7219 | both: 7219
Exclusions from primary: {'incomplete_initial_history': 5, 'unidentified_lane_opponent;global_observed_rate_unavailable:unidentified_lane_opponent': 4, 'global_observed_rate_unavailable:not_published_in_table': 2}
Exclusions from paired: {'na_observed_rate_unavailable:not_published_in_table': 13, 'incomplete_initial_history': 5, 'unidentified_lane_opponent;global_observed_rate_unavailable:unidentified_lane_opponent;na_observed_rate_unavailable:unidentified_lane_opponent': 4, 'global_observed_rate_unavailable:not_published_in_table;na_observed_rate_unavailable:not_published_in_table': 2}
Validation: {'source_keys_outcomes_queues_and_timing': 'PASS', 'regional_lookup_filters_units_and_cached_provenance': 'PASS', 'all_original_observed_keys_and_outcomes_preserved': 'PASS', 'no_join_row_multiplication_or_duplicate_keys': 'PASS', 'each_observed_pick_occurs_once': 'PASS', 'unplayed_candidates_have_no_outcome': 'PASS', 'strict_completio

The primary subset requires a known lane opponent and a verified Global observed rate and excludes the incomplete-initial-history player. The paired subset is drawn from primary and additionally requires a verified NA observed rate. No external-game minimum, champion-experience minimum, alternative-count requirement or new OTP cutoff is applied. The full master retains all exclusions with reasons.

Use observed tables for outcome modeling and the separate candidate table for descriptive comparisons. No ready-made predictor matrix is supplied. Collection/rank metadata and postgame audit fields remain for audit and later explicit feature selection.

In [5]:
OUTPUT_DIR, preparation_metadata = prep.save_outputs(data, tables, validation, cleanup)
print('Preparation completed:', OUTPUT_DIR)
print('Unresolved collection pages:', len(preparation_metadata['unresolved_pages']))
print('Unresolved collection keys:', sum(row['planned_keys'] for row in preparation_metadata['unresolved_pages']))
print('Saved five Parquet datasets, CSV equivalents, data dictionary, README and preparation_metadata.json.')


Preparation completed: /Users/minhho-hoang/Documents/riot_proj/riot-game-prediction/hlr_model/data/final/20261020T180254240509Z
Unresolved collection pages: 12
Unresolved collection keys: 121
Saved five Parquet datasets, CSV equivalents, data dictionary, README and preparation_metadata.json.
